In [ ]:
import numpy as np
from keras.preprocessing.image import ImageDataGenerator
from keras.applications import VGG16
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

In [ ]:
# Load VGG-16 model without top (fully connected layers)
vgg_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

In [ ]:
# Define function to extract features using VGG-16
def extract_features(generator, model):
    features = model.predict(generator, verbose=1)
    return features

In [ ]:
# Define image size and batch size
image_size = (224, 224)
batch_size = 32

In [ ]:
# Directory paths for train and test data
train_dir = "Processed_Dataset_2/train"
test_dir = "Processed_Dataset_2/test"

In [ ]:
# Create data generators for train and test data
train_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode=None,  # No labels are returned
    shuffle=False  # Important: Keep the order of the images
)

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode=None,
    shuffle=False
)

In [ ]:
# Extract features for train and test data
train_features = extract_features(train_generator, vgg_model)
test_features = extract_features(test_generator, vgg_model)

In [ ]:
# Flatten features
train_features_flatten = np.reshape(train_features, (train_features.shape[0], -1))
test_features_flatten = np.reshape(test_features, (test_features.shape[0], -1))

In [ ]:
# Train XGBoost classifier
xgb_classifier = XGBClassifier(n_estimators=100, random_state=42)
xgb_classifier.fit(train_features_flatten, train_generator.classes)

In [ ]:
# Predict using XGBoost classifier
predictions = xgb_classifier.predict(test_features_flatten)

In [ ]:
# Calculate accuracy
accuracy = accuracy_score(test_generator.classes, predictions)
print("Accuracy:", accuracy)